In [1]:
GRAY_TO_CLASS = {0: -1, 64: 0, 128: 1, 192: 2, 255: 3}  # -1 = background, ignorado
CLASS_TO_GRAY = {0: 64, 1: 128, 2: 192, 3: 255}
CLASS_NAMES = ["No_Proliferativo", "Proliferativo", "Esclerosado", "Excluido"]
NUM_CLASSES = 4

CONFIG = {
    # Paths — leer desde Entradas/ (tiles crudos + máscaras manuales)
    'images_dir': 'Salidas/Tiles_UNet',              # <slide>/images/*.png — igual que U-Net
    'masks_dir':  'Salidas/Tiles_UNet',              # <slide>/masks/*_mask.png — máscaras multiclase
    'output_dir': 'Salidas/Clasificador',

    # Split — MISMO que U-Net
    'train_size': 0.70,
    'val_size':   0.15,
    'seed':       42,

    # Crop/reconstruction
    'input_size':   224,
    'mask_size':    224,
    'margin_ratio': 0.25,
    'min_area_px':  1500,
    'min_distance': 15,

    # Model
    'backbone':    'efficientnet_b0',
    'input_mode':  'rgb_mask_manual',
    'num_classes': 4,
    'pretrained':  True,

    # Training
    'max_epochs':      60,
    'batch_size':  32,
    'lr':          1e-3,
    'weight_decay': 1e-4,
    'warmup_epochs': 5,
    'use_amp':     True,
    'early_stopping_patience': 10,
    'early_stopping_min_delta': 1e-4,

    # Sampling & Loss strategy
    'sampling_strategy': 'sampler_balanced_ce_unweighted',
}

In [2]:
# Standard library
import os
import json
import random
import warnings
from pathlib import Path
from typing import Tuple

# Scientific computing / ML
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import albumentations as A
import timm
import matplotlib.pyplot as plt

# Data loading and metrics
from PIL import Image, ImageFile
from scipy.ndimage import label as scipy_label, distance_transform_edt
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from skimage.feature import peak_local_max as _plm
from skimage.measure import regionprops
from skimage.segmentation import watershed as _watershed
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm

# Allow PIL to load truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# --- CUDA performance tuning (T4 Tensor Cores) ---
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True          # auto-tune conv algorithms
    torch.backends.cuda.matmul.allow_tf32 = True   # TF32 for matmul (T4 supports it)
    torch.backends.cudnn.allow_tf32 = True         # TF32 for cudnn convolutions
    print(f"cudnn.benchmark: {torch.backends.cudnn.benchmark}")
    print(f"CUDA matmul TF32: {torch.backends.cuda.matmul.allow_tf32}")
    print(f"cudnn TF32: {torch.backends.cudnn.allow_tf32}")
    print(f"PYTORCH_CUDA_ALLOC_CONF: expandable_segments:True")


# --- Preprocessing Transforms (Reinhard Normalization + Z-score) ---

class ReinhardNormalize:
    """Reinhard stain normalization in LAB color space for consistency across slides."""

    def __init__(self, target_stats: dict):
        self.target = target_stats

    @staticmethod
    def _get_tissue_mask(img_bgr: np.ndarray) -> np.ndarray:
        """Isolate tissue pixels from background and artifacts via luminance and saturation thresholds."""
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
        hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
        mask = (lab[:, :, 0] < 230) & (hsv[:, :, 1] > 10)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
        mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE, kernel)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        return mask.astype(bool)

    @staticmethod
    def compute_template_stats(image_paths: list, n_samples: int = 200) -> dict:
        """Compute median LAB statistics from tissue pixels to define normalization target."""
        paths_list = list(image_paths)[:n_samples]
        sample = random.sample(paths_list, min(n_samples, len(paths_list)))
        all_stats = []
        
        for p in sample:
            img = cv2.imread(str(p))
            if img is None:
                continue
            tissue = ReinhardNormalize._get_tissue_mask(img)
            if tissue.sum() < 100:
                continue
            lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).astype(np.float32)
            stats = ([lab[..., c][tissue].mean() for c in range(3)] +
                     [lab[..., c][tissue].std()  for c in range(3)])
            all_stats.append(stats)
        
        if not all_stats:
            return {'mean_L': 50, 'mean_a': 128, 'mean_b': 128,
                    'std_L': 10, 'std_a': 10, 'std_b': 10}
        
        arr = np.array(all_stats)
        keys = ['mean_L', 'mean_a', 'mean_b', 'std_L', 'std_a', 'std_b']
        return {k: float(np.median(arr[:, i])) for i, k in enumerate(keys)}

    def __call__(self, img_bgr: np.ndarray) -> np.ndarray:
        """Normalize image to match template statistics, preserving background pixels."""
        tissue = self._get_tissue_mask(img_bgr)
        if tissue.sum() < 100:
            return img_bgr
        
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
        
        src = {c: (lab[..., i][tissue].mean(), lab[..., i][tissue].std())
               for i, c in enumerate(['L', 'a', 'b'])}
        
        result = lab.copy()
        for i, c in enumerate(['L', 'a', 'b']):
            m, s = src[c]
            result[..., i] = ((lab[..., i] - m) *
                              (self.target[f'std_{c}'] / (s + 1e-5)) +
                              self.target[f'mean_{c}'])
        
        result[~tissue] = lab[~tissue]
        result = np.clip(result, 0, 255).astype(np.uint8)
        return cv2.cvtColor(result, cv2.COLOR_LAB2BGR)


def compute_channel_stats(
    image_paths: list,
    n_samples: int = 200,
    reinhard_norm=None,
    std_floor: float = 0.03,
) -> Tuple[list, list]:
    """Compute weighted per-channel RGB stats from tissue pixels for Z-score normalization."""
    paths_list = list(image_paths)
    sample = random.sample(paths_list, min(n_samples, len(paths_list)))
    
    tile_means, tile_vars, tile_counts = [], [], []
    skipped_count = 0
    for p in sample:
        img_bgr = cv2.imread(str(p))
        if img_bgr is None:
            skipped_count += 1
            continue
        tissue = ReinhardNormalize._get_tissue_mask(img_bgr)
        if tissue.sum() < 100:
            skipped_count += 1
            continue

        if reinhard_norm is not None:
            img_bgr = reinhard_norm(img_bgr)

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        pixels = img_rgb[tissue]
        tile_means.append(pixels.mean(0))
        tile_vars.append(pixels.var(0))
        tile_counts.append(tissue.sum())
    
    if not tile_means:
        print(f"  ⚠️  WARNING: No valid tissue pixels found in {len(sample)} samples!")
        print(f"      {skipped_count} samples were skipped (tissue.sum() < 100 or read failed)")
        print(f"      Falling back to default values [0.5, 0.5, 0.5] and [0.2, 0.2, 0.2]")
        print(f"      This will make Z-score normalization INEFFECTIVE.")
        return [0.5, 0.5, 0.5], [0.2, 0.2, 0.2]
    
    print(f"  ✓ Computed stats from {len(tile_means)} valid samples (skipped {skipped_count})")
    
    means_arr = np.array(tile_means)
    vars_arr = np.array(tile_vars)
    counts_arr = np.array(tile_counts, dtype=np.float64)
    w = counts_arr / counts_arr.sum()
    
    mean = (means_arr * w[:, None]).sum(0)
    var = ((vars_arr + (means_arr - mean) ** 2) * w[:, None]).sum(0)
    std = np.sqrt(var)
    std = np.maximum(std, std_floor)
    return mean.tolist(), std.tolist()

def _collect_image_tiles(images_dir: str) -> list:
    """Collect input PNG tiles while excluding masks and generated mask files."""
    images_dir = Path(images_dir)
    image_paths = sorted(images_dir.glob('*/images/*.png'))

    # Fallback for flat/custom datasets: include PNGs except anything inside a masks folder
    # or files already named *_mask.png.
    if not image_paths:
        image_paths = sorted(
            p for p in images_dir.rglob('*.png')
            if 'masks' not in p.relative_to(images_dir).parts
            and not p.stem.endswith('_mask')
        )

    return image_paths


def _slide_name_from_image_path(image_path, images_dir) -> str:
    """Infer the biopsy/slide name from a tile path under <root>/<slide>/images/*.png."""
    image_path = Path(image_path)
    images_dir = Path(images_dir)
    try:
        rel = image_path.relative_to(images_dir)
        if len(rel.parts) >= 3 and rel.parts[1] == 'images':
            return rel.parts[0]
    except ValueError as e:
        warnings.warn(f'Path format issue for {image_path}: {e}')
        return None

    if image_path.parent.name == 'images' and image_path.parent.parent.name:
        return image_path.parent.parent.name
    return image_path.parent.name or image_path.stem


def _group_images_by_biopsy(image_paths: list, images_dir: str) -> dict:
    """Group image paths by biopsy/slide folder, supporting canonical and flat layouts."""
    images_dir = Path(images_dir)
    biopsias_dict = {}
    for img_path in image_paths:
        biopsia = _slide_name_from_image_path(img_path, images_dir)
        biopsias_dict.setdefault(biopsia, []).append(img_path)
    return biopsias_dict


def _safe_train_val_test_split(
    biopsias_list: list,
    train_size: float = 0.70,
    val_size: float = 0.15,
    seed: int = 42,
) -> Tuple[list, list, list]:
    """Split biopsy IDs without crashing on tiny datasets."""
    test_size = 1.0 - train_size - val_size
    assert test_size >= 0, "train_size + val_size must be <= 1.0"

    biopsias_list = list(biopsias_list)
    if not biopsias_list:
        return [], [], []

    # sklearn.train_test_split raises on very small lists. Keep deterministic, leak-free
    # biopsy-level splits and prefer having train data in quick/smoke-test datasets.
    if len(biopsias_list) == 1:
        return biopsias_list, [], []
    if len(biopsias_list) == 2:
        rng = random.Random(seed)
        shuffled = biopsias_list[:]
        rng.shuffle(shuffled)
        return [shuffled[0]], [shuffled[1]], []

    if test_size > 0:
        train_val_biopsias, test_biopsias = train_test_split(
            biopsias_list,
            test_size=test_size,
            random_state=seed,
        )
    else:
        train_val_biopsias = biopsias_list
        test_biopsias = []

    if val_size > 0 and len(train_val_biopsias) > 1:
        val_fraction = val_size / (train_size + val_size)
        train_biopsias, val_biopsias = train_test_split(
            train_val_biopsias,
            test_size=val_fraction,
            random_state=seed + 1,
        )
    else:
        train_biopsias = train_val_biopsias
        val_biopsias = []

    return train_biopsias, val_biopsias, test_biopsias


def split_biopsias(
    images_dir: str,
    train_size: float = 0.70,
    val_size: float = 0.15,
    seed: int = 42,
) -> Tuple[list, list, list, dict]:
    """Groups biopsias into train/val/test to prevent data leakage at slide level."""
    images_dir = Path(images_dir)
    all_images = _collect_image_tiles(images_dir)

    if not all_images:
        raise ValueError(f"No PNG image tiles found in {images_dir}. Expected files under */images/*.png")

    biopsias_dict = _group_images_by_biopsy(all_images, images_dir)
    train_biopsias, val_biopsias, test_biopsias = _safe_train_val_test_split(
        list(biopsias_dict.keys()),
        train_size=train_size,
        val_size=val_size,
        seed=seed,
    )

    return train_biopsias, val_biopsias, test_biopsias, biopsias_dict



def kfold_biopsy_split(biopsias: list, k: int = 5) -> list:
    """
    K-fold cross-validation split at biopsy level.
    Returns k folds, each with train/val/test biopsies.
    """
    from sklearn.model_selection import KFold
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    folds = []
    for train_idx, test_idx in kf.split(biopsias):
        train_b = [biopsias[i] for i in train_idx]
        val_b = train_b[:max(1, len(train_b)//5)]   # 20% of train for val
        train_b = train_b[len(val_b):]
        test_b = [biopsias[i] for i in test_idx]
        folds.append({'train': train_b, 'val': val_b, 'test': test_b})
    return folds


def leave_one_biopsy_out_split(biopsias: list) -> list:
    """
    Leave-one-biopsy-out cross-validation.
    Each biopsy is test once; rest divided 85/15 into train/val.
    """
    folds = []
    for i, test_b in enumerate(biopsias):
        remaining = [b for j, b in enumerate(biopsias) if j != i]
        val_b = remaining[:max(1, len(remaining)//6)]
        train_b = remaining[len(val_b):]
        folds.append({'train': train_b, 'val': val_b, 'test': [test_b]})
    return folds


def load_rgb_image(image_path: str) -> np.ndarray:
    """Load an RGB uint8 image with PIL tolerance for truncated files."""
    try:
        return np.array(Image.open(str(image_path)).convert('RGB'))
    except Exception as exc:
        raise RuntimeError(f"Failed to load image {image_path}: {exc}") from exc


def load_binary_mask(mask_path: str) -> np.ndarray:
    """Load a grayscale mask and binarize all non-zero classes as glomerulus."""
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise RuntimeError(f"Failed to load mask: {mask_path}")
    return (mask > 0).astype(np.uint8)



def load_gray_mask(mask_path: str) -> np.ndarray:
    """Load a grayscale mask preserving multiclass values (0/64/128/192/255)."""
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise RuntimeError(f"Failed to load mask: {mask_path}")
    return mask
def preprocess_rgb_image(
    img_rgb: np.ndarray,
    reinhard_norm=None,
    channel_means: list = None,
    channel_stds: list = None,
) -> np.ndarray:
    """Apply the same RGB -> Reinhard -> RGB/255 -> Z-score preprocessing used by the dataset."""
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    if reinhard_norm is not None:
        img_bgr = reinhard_norm(img_bgr)

    img_float = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    if channel_means is not None and channel_stds is not None:
        means = np.asarray(channel_means, dtype=np.float32)
        stds = np.asarray(channel_stds, dtype=np.float32)
        img_float = (img_float - means) / (stds + 1e-6)
    return img_float


def tensor_from_preprocessed_rgb(img_float: np.ndarray, device: torch.device = None) -> torch.Tensor:
    """Convert preprocessed HWC RGB float image to a BCHW float tensor."""
    tensor = torch.from_numpy(np.transpose(img_float, (2, 0, 1))).float().unsqueeze(0)
    return tensor.to(device) if device is not None else tensor


def make_mask_overlay(img_rgb: np.ndarray, mask_binary: np.ndarray, color=(255, 0, 0), alpha: float = 0.4) -> np.ndarray:
    """Blend a binary mask over an RGB image."""
    overlay = img_rgb.copy().astype(np.float32)
    overlay[mask_binary.astype(bool)] = color
    return cv2.addWeighted(img_rgb, 1 - alpha, overlay.astype(np.uint8), alpha, 0)


# ============================================================================
# Grayscale pixel values that correspond to glomerulus classes (any > 0 in practice)
# 64=No_Proliferativo, 128=Proliferativo, 192=Esclerosado, 255=Excluido/Excluyente
# Excluido (255) is INTENTIONALLY mapped to Glomerulus class 1 for binary segmentation
# ============================================================================
class GlomeruliDataset(Dataset):
    """Loads paired image-mask glomeruli tiles with online preprocessing and augmentation."""

    def __init__(
        self,
        images_dir: str,
        masks_dir: str = None,
        split: str = 'train',
        biopsias: list = None,
        biopsias_dict: dict = None,
        reinhard_norm=None,
        channel_means: list = None,
        channel_stds: list = None,
        train_size: float = 0.70,
        val_size: float = 0.15,
        seed: int = 42,
        transforms=None,
    ):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir) if masks_dir is not None else Path(images_dir)
        self.split = split
        self.transforms = transforms
        self.reinhard_norm = reinhard_norm
        self.channel_means = channel_means if channel_means is not None else [0.5, 0.5, 0.5]
        self.channel_stds = channel_stds if channel_stds is not None else [0.2, 0.2, 0.2]

        assert split in {'train', 'val', 'test'}, f"Invalid split: {split}"
        assert self.images_dir.exists(), f"Images dir not found: {self.images_dir}"
        assert self.masks_dir.exists(), f"Masks dir not found: {self.masks_dir}"

        if biopsias is not None and biopsias_dict is not None:
            selected_biopsias = biopsias
            self.biopsias_dict = biopsias_dict
        else:
            train_biopsias, val_biopsias, test_biopsias, self.biopsias_dict = split_biopsias(
                self.images_dir,
                train_size=train_size,
                val_size=val_size,
                seed=seed,
            )
            selected_biopsias = {
                'train': train_biopsias,
                'val': val_biopsias,
                'test': test_biopsias,
            }[split]

        self.image_paths = []
        for biopsia in selected_biopsias:
            self.image_paths.extend(self.biopsias_dict.get(biopsia, []))

        self.image_paths = sorted(self.image_paths)

        paired = []
        missing = []
        for img_path in self.image_paths:
            mask_path = self._get_mask_path(img_path)
            if mask_path.exists():
                paired.append((img_path, mask_path))
            else:
                missing.append((img_path, mask_path))

        if missing:
            warnings.warn(
                f"Found {len(missing)} images without corresponding masks. "
                f"These will be skipped. First few: {missing[:3]}"
            )

        if not paired:
            raise ValueError("No valid image-mask pairs found after checking.")

        self.image_paths, self.mask_paths = zip(*paired)
        self.image_paths = list(self.image_paths)
        self.mask_paths = list(self.mask_paths)
        
        # Caches used by the train sampler/audits
        self._positive_flags = None
        self._tile_metadata = None
        self._annotation_tiles_by_key = None

    def _get_mask_path(self, image_path: Path) -> Path:
        """Convert image tile path to its mask path, supporting canonical and flat layouts."""
        rel = image_path.relative_to(self.images_dir)
        parts = list(rel.parts)
        stem = Path(parts[-1]).stem

        if len(parts) >= 3 and parts[1] == 'images':
            parts[1] = 'masks'
            parts[-1] = f"{stem}_mask.png"
            return self.masks_dir / Path(*parts)

        # Flat/custom fallback: first try <masks_dir>/<stem>_mask.png, then masks/<stem>_mask.png.
        flat_mask = self.masks_dir / f"{stem}_mask.png"
        if flat_mask.exists():
            return flat_mask
        return self.masks_dir / 'masks' / f"{stem}_mask.png"

    def get_positive_flags(self) -> list:
        """Return list of bools: True if tile mask contains at least one glomerulus pixel.
        
        Used by WeightedRandomSampler. Reads mask files once at dataset init time.
        Masks are small enough (1024x1024 uint8 = 1MB) that this is feasible.
        Caches result to avoid double scan.
        """
        if self._positive_flags is not None:
            return self._positive_flags
        
        flags = []
        for mask_path in self.mask_paths:
            try:
                flags.append(bool(np.any(load_binary_mask(mask_path))))
            except RuntimeError:
                flags.append(False)
        self._positive_flags = flags
        return flags

    def _load_annotation_tiles_by_key(self) -> dict:
        """Index per-slide annotations.json entries by (slide_folder, tile image path)."""
        if self._annotation_tiles_by_key is not None:
            return self._annotation_tiles_by_key

        tiles_by_key = {}
        for ann_path in sorted(self.images_dir.glob('*/annotations.json')):
            slide_folder = ann_path.parent.name
            try:
                with ann_path.open('r', encoding='utf-8') as f:
                    annotations = json.load(f)
            except Exception as exc:
                warnings.warn(f"Could not read annotations metadata from {ann_path}: {exc}")
                continue

            slide_name = annotations.get('slide') or slide_folder
            for tile in annotations.get('tiles', []):
                image_rel = tile.get('image')
                if not image_rel:
                    continue

                image_rel = str(Path(image_rel).as_posix())
                meta = dict(tile)
                meta['slide'] = slide_name
                meta['slide_folder'] = slide_folder
                meta['annotations_path'] = str(ann_path)

                # Support both the folder name and the slide name in case they differ.
                tiles_by_key[(slide_folder, image_rel)] = meta
                tiles_by_key[(slide_name, image_rel)] = meta

        self._annotation_tiles_by_key = tiles_by_key
        return tiles_by_key

    def get_tile_metadata(self, idx: int) -> dict:
        """Return annotations.json metadata for a dataset tile, or {} if unavailable."""
        if self._tile_metadata is None:
            tiles_by_key = self._load_annotation_tiles_by_key()
            metadata = []
            for image_path in self.image_paths:
                try:
                    rel = image_path.relative_to(self.images_dir)
                    slide_folder = rel.parts[0]
                    image_rel = Path(*rel.parts[1:]).as_posix()
                except Exception:
                    metadata.append({})
                    continue

                metadata.append(tiles_by_key.get((slide_folder, image_rel), {}))

            self._tile_metadata = metadata

        return self._tile_metadata[idx] if 0 <= idx < len(self._tile_metadata) else {}

    def get_sampling_weights(
        self,
        secondary_factor: float = 0.35,
        duplicate_aware: bool = True,
    ) -> Tuple[torch.Tensor, dict]:
        """Compute train-sampling weights with optional duplicate-aware glomerulus balancing."""
        positive_flags = self.get_positive_flags()
        n_positive = int(sum(positive_flags))
        n_negative = int(len(positive_flags) - n_positive)

        report = {
            'mode': 'simple',
            'n_positive': n_positive,
            'n_negative': n_negative,
            'num_unique_glomeruli': 0,
            'primary_links': 0,
            'secondary_links': 0,
            'unmatched_positive_tiles': 0,
        }

        if n_positive == 0 or n_negative == 0:
            return None, report

        def simple_weights(mode: str = 'simple'):
            report['mode'] = mode
            weight_pos = 1.0 / n_positive
            weight_neg = 1.0 / n_negative
            return torch.tensor(
                [weight_pos if flag else weight_neg for flag in positive_flags],
                dtype=torch.float32,
            ), report

        if not duplicate_aware:
            return simple_weights('simple')

        glomerulus_entries = {}
        for idx, is_positive in enumerate(positive_flags):
            if not is_positive:
                continue

            meta = self.get_tile_metadata(idx)
            glomeruli = meta.get('glomeruli') if meta else None
            if not glomeruli:
                continue

            slide = meta.get('slide') or meta.get('slide_folder') or _slide_name_from_image_path(
                self.image_paths[idx], self.images_dir
            )
            for glom in glomeruli:
                glom_id = glom.get('id')
                if glom_id is None:
                    continue

                role = str(glom.get('role', 'primary')).lower()
                try:
                    coverage = max(float(glom.get('coverage_pct', 0.0)), 0.0) / 100.0
                except (TypeError, ValueError):
                    coverage = 0.0

                role_factor = secondary_factor if role == 'secondary' else 1.0
                raw_weight = max(coverage, 1e-6) * role_factor
                key = (str(slide), str(glom_id))
                glomerulus_entries.setdefault(key, []).append((idx, raw_weight, role))

                if role == 'secondary':
                    report['secondary_links'] += 1
                else:
                    report['primary_links'] += 1

        if not glomerulus_entries:
            warnings.warn(
                "Duplicate-aware sampler requested, but no usable glomerulus metadata was found. "
                "Falling back to simple positive/negative balancing."
            )
            return simple_weights('fallback-simple-no-annotations')

        positive_mass = np.zeros(len(positive_flags), dtype=np.float64)
        for entries in glomerulus_entries.values():
            total = sum(raw for _, raw, _ in entries)
            if total <= 0:
                continue
            for idx, raw, _ in entries:
                positive_mass[idx] += raw / total

        unmatched_positive_tiles = 0
        for idx, is_positive in enumerate(positive_flags):
            if is_positive and positive_mass[idx] <= 0:
                # Keep mask-positive tiles with missing/partial metadata trainable.
                positive_mass[idx] = 1.0
                unmatched_positive_tiles += 1

        total_positive_mass = float(positive_mass.sum())
        if total_positive_mass <= 0:
            warnings.warn(
                "Duplicate-aware sampler produced zero positive mass. "
                "Falling back to simple positive/negative balancing."
            )
            return simple_weights('fallback-simple-zero-positive-mass')

        weights = np.zeros(len(positive_flags), dtype=np.float64)
        for idx, is_positive in enumerate(positive_flags):
            if is_positive:
                weights[idx] = positive_mass[idx] / total_positive_mass
            else:
                weights[idx] = 1.0 / n_negative

        report.update({
            'mode': 'duplicate-aware',
            'num_unique_glomeruli': len(glomerulus_entries),
            'unmatched_positive_tiles': unmatched_positive_tiles,
            'positive_weight_sum': float(weights[np.array(positive_flags, dtype=bool)].sum()),
            'negative_weight_sum': float(weights[~np.array(positive_flags, dtype=bool)].sum()),
            'secondary_factor': secondary_factor,
        })
        return torch.tensor(weights, dtype=torch.float32), report

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """Load RGB tile -> Reinhard normalization -> augment -> Z-score -> tensors."""
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        img_rgb = load_rgb_image(img_path)
        mask_binary = load_binary_mask(mask_path)

        if self.reinhard_norm is not None:
            img_bgr = self.reinhard_norm(cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        if self.transforms is not None:
            augmented = self.transforms(image=img_rgb, mask=mask_binary)
            img_rgb = augmented['image']
            mask_binary = augmented['mask']

        img_float = preprocess_rgb_image(
            img_rgb,
            reinhard_norm=None,  # already applied before augmentation when configured
            channel_means=self.channel_means,
            channel_stds=self.channel_stds,
        )
        img_tensor = torch.from_numpy(np.transpose(img_float, (2, 0, 1))).float()
        mask_tensor = torch.from_numpy(mask_binary.astype(np.int64)).long()

        return img_tensor, mask_tensor

# Augmentation hyperparameters (documented magic numbers)
AUGMENTATION_CONFIG = {
    'elastic': {'alpha': 120, 'sigma': 6.0},
    'he_stain': {
        'intensity_scale': (0.8, 1.2),
        'intensity_shift': (-0.1, 0.1),
    },
    'color_jitter': {
        'brightness': 0.2,
        'contrast': 0.2,
        'saturation': 0.2,
        'hue': 0.05,
    },
    'gaussian_noise': {'std_range': (0.01, 0.05)},
}

def get_transforms(size: int = 1024, config: dict | None = None):
    """
    Get training augmentations with tunable hyperparameters.
    
    Args:
        size: Image size
        config: Augmentation config dict. Defaults to AUGMENTATION_CONFIG
    
    Notes:
        Reinhard and Z-score normalization happen in GlomeruliDataset.__getitem__,
        not here. Train transforms run on RGB uint8 images before Z-score; validation
        uses an explicit no-op transform for symmetry.
    """
    if config is None:
        config = AUGMENTATION_CONFIG
    
    train_augment = A.Compose([
        # Geometric augmentations — applied to both image AND mask
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.75),
        A.Transpose(p=0.5),
        A.Rotate(limit=15, p=0.3),
        A.ShiftScaleRotate(scale_limit=0.15, rotate_limit=15, shift_limit=0.1, p=0.5),
        
        # Elastic deformations — simulate tissue preparation artifacts
        A.ElasticTransform(alpha=120, sigma=120 * 0.05, p=0.3),
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.2),
        
        # H&E stain augmentation — simulates staining variability between biopsies
        # ImageOnlyTransform: applied to image only, mask is unchanged
        A.HEStain(
            method='random_preset',
            intensity_scale_range=(0.8, 1.2),
            intensity_shift_range=(-0.1, 0.1),
            augment_background=False,
            p=0.4,
        ),

        # Color augmentations — simulate stain variation between slides
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.3),
        
        # Noise — simulate scanner artifacts
        A.GaussNoise(std_range=(0.01, 0.05), p=0.2),
        
        # Regularization via occlusion (use fill=128 to avoid NaN in BatchNorm)
        A.CoarseDropout(
            num_holes_range=(1, 8),
            hole_height_range=(32, 64),
            hole_width_range=(32, 64),
            fill=128,
            p=0.2,
        ),
    ], additional_targets={'mask': 'mask'})
    
    val_transform = A.Compose([])
    
    return train_augment, val_transform

# ===== CENTRALIZED POST-PROCESSING FUNCTION =====
# This function is used by threshold tuning, crop extraction, and reconstruction viz
# to ensure CONSISTENCY across all stages

def postprocess_prob_to_instances(
    prob_map,           # H×W float32 probability map (0-1)
    threshold=0.4,
    min_area_px=1500,
    min_distance=15,    # distance between watershed peaks
):
    """
    Consistent post-processing pipeline for all glomerulus detection tasks.
    
    Args:
        prob_map: H×W float32 array in [0, 1]
        threshold: decision threshold
        min_area_px: minimum instance area in pixels
        min_distance: minimum distance between watershed peaks
    
    Returns:
        list of skimage regionprops objects (instances)
    """
    # Step 1: Binary mask from probability
    binary_mask = (prob_map > threshold).astype(np.uint8)
    
    # Step 2: Morphological cleaning (close → open)
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, kernel_close)
    binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, kernel_open)
    
    # Step 3: Watershed-based instance separation
    if binary_mask.sum() < min_area_px:
        return []
    
    distance = distance_transform_edt(binary_mask)
    coords = _plm(distance, min_distance=min_distance, labels=binary_mask)
    
    if len(coords) == 0:
        # Single large region
        coords = np.array([[binary_mask.shape[0]//2, binary_mask.shape[1]//2]])
    
    marker_mask = np.zeros_like(binary_mask, dtype=bool)
    marker_mask[tuple(coords.T)] = True
    markers, _ = scipy_label(marker_mask)
    
    labels_map = _watershed(-distance, markers, mask=binary_mask)
    props = regionprops(labels_map)
    
    # Step 4: Filter by minimum area
    return [p for p in props if p.area >= min_area_px]


c:\Users\proyecto_final\Documents\Proyecto_Final_Glomerulos\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
GPU: Tesla T4
VRAM: 17.00 GB
cudnn.benchmark: True
CUDA matmul TF32: True
cudnn TF32: True
PYTORCH_CUDA_ALLOC_CONF: expandable_segments:True


In [3]:
def perturb_mask(binary_mask: np.ndarray) -> np.ndarray:
    """Perturba la máscara binaria para simular error de U-Net en entrenamiento."""
    ops = random.choices(['dilate', 'erode', 'close', 'open', 'shift', 'none'], k=1)[0]
    k = random.choice([3, 5])
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    if ops == 'dilate':
        return cv2.dilate(binary_mask.astype(np.uint8), kernel).astype(bool)
    elif ops == 'erode':
        return cv2.erode(binary_mask.astype(np.uint8), kernel).astype(bool)
    elif ops == 'close':
        return cv2.morphologyEx(binary_mask.astype(np.uint8), cv2.MORPH_CLOSE, kernel).astype(bool)
    elif ops == 'open':
        return cv2.morphologyEx(binary_mask.astype(np.uint8), cv2.MORPH_OPEN, kernel).astype(bool)
    elif ops == 'shift':
        dx, dy = random.randint(-5, 5), random.randint(-5, 5)
        M = np.float32([[1, 0, dx], [0, 1, dy]])
        shifted = cv2.warpAffine(binary_mask.astype(np.uint8), M, binary_mask.shape[::-1])
        return shifted.astype(bool)
    return binary_mask

In [4]:
def build_glomerulus_manifest(
    images_dir: str,
    masks_dir: str,
    split_biopsias_result: dict,  # {'train': [...], 'val': [...], 'test': [...]}
    min_area_px: int = 1500,
    min_distance: int = 15,
    output_csv: str = None,
) -> pd.DataFrame:
    """
    Enumerate ALL glomerulus instances in the dataset (train+val+test).
    Each row is one glomerulus instance with its metadata, bbox, and class.
    
    Algorithm:
    1. For each split ('train', 'val', 'test'):
       - For each biopsia in split:
         - Enumerate tiles: {images_dir}/{biopsia}/images/*.png
         - For EACH tile:
           a. Load mask_gray = load_binary_mask(mask_path, binary=False)
              (PNG uint8 with values 0/64/128/192/255)
           b. Binarize: binary_mask = (mask_gray > 0).astype(float)
           c. Run instance detection: instances = postprocess_prob_to_instances(...)
           d. For EACH instance:
              - Extract bbox: instance.bbox → (r1, c1, r2, c2)
              - Extract region from mask_gray: mask_crop = mask_gray[r1:r2, c1:c2]
              - Find dominant (most frequent) value: dominant = bincount(mask_crop.flatten()).argmax()
              - Map to class: class_id = GRAY_TO_CLASS[dominant] (skip if -1/background)
              - Add row with: slide_id, tile_path, mask_path, instance_id, class_name, 
                             class_id, gray_value, bbox coords, split
    
    2. Return DataFrame with all rows
    3. If output_csv not None, save to CSV
    
    Args:
        images_dir: Root directory with {biopsia}/images/*.png structure
        masks_dir: Root directory with {biopsia}/masks/*_mask.png structure
        split_biopsias_result: dict with keys 'train', 'val', 'test' → list of biopsia names
        min_area_px: Minimum instance area in pixels (default 1500)
        min_distance: Minimum distance between watershed peaks (default 15)
        output_csv: If not None, save manifest to this CSV path
    
    Returns:
        pd.DataFrame with columns:
        - slide_id: biopsia name
        - tile_path: absolute path to tile PNG
        - mask_path: absolute path to mask PNG
        - instance_id: index within tile (0, 1, 2, ...)
        - class_name: CLASS_NAMES[class_id]
        - class_id: int 0-3
        - gray_value: dominant gray value (64, 128, 192, 255)
        - bbox_r1, bbox_c1, bbox_r2, bbox_c2: bounding box coordinates
        - split: 'train'/'val'/'test'
    
    Validation:
        - DataFrame has ~100s-1000s rows (1 per glomerulus, not per tile)
        - All columns present
        - No NaN in critical columns
        - Split distribution: ~70% train, ~15% val, ~15% test (approximate)
    """
    images_dir = Path(images_dir)
    masks_dir = Path(masks_dir)
    
    manifest_rows = []
    
    # Iterate through all splits
    for split_name in ['train', 'val', 'test']:
        biopsias = split_biopsias_result.get(split_name, [])
        
        for biopsia in biopsias:
            # Find all tiles in this biopsia: {images_dir}/{biopsia}/images/*.png
            tiles_glob = sorted((images_dir / biopsia / 'images').glob('*.png'))
            
            for tile_path in tiles_glob:
                # Construct corresponding mask path
                tile_stem = tile_path.stem
                mask_path = masks_dir / biopsia / 'masks' / f'{tile_stem}_mask.png'
                
                # Check if mask exists
                if not mask_path.exists():
                    warnings.warn(f"Mask not found for tile {tile_path}, skipping.")
                    continue
                
                # Load mask as grayscale (uint8, not binarized)
                try:
                    mask_gray = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
                    if mask_gray is None:
                        warnings.warn(f"Failed to load mask: {mask_path}")
                        continue
                except Exception as e:
                    warnings.warn(f"Error loading mask {mask_path}: {e}")
                    continue
                
                # Binarize: any non-zero value is glomerulus
                binary_mask = (mask_gray > 0).astype(float)
                
                # Run instance detection
                instances = postprocess_prob_to_instances(
                    binary_mask,
                    threshold=0.5,
                    min_area_px=min_area_px,
                    min_distance=min_distance,
                )
                
                # Get label image for instance indexing
                label_im, num_labels = scipy_label(binary_mask)
                
                # For each detected instance, extract metadata
                for instance_id, instance in enumerate(instances):
                    # Extract bounding box
                    r1, c1, r2, c2 = instance.bbox
                    
                    # Get pixels for THIS instance only (using label_im directly)
                    # Instance labels start at 1, so we use instance_id + 1
                    instance_label = instance_id + 1
                    instance_pixels = mask_gray[label_im == instance_label]
                    if len(instance_pixels) == 0:
                        continue
                    valid_pixels = instance_pixels[instance_pixels > 0]
                    if len(valid_pixels) == 0:
                        continue
                    dominant = int(np.bincount(valid_pixels).argmax())
                    
                    # Map to class
                    class_id = GRAY_TO_CLASS.get(dominant, -1)
                    
                    # Skip background/unknown classes
                    if class_id == -1:
                        continue
                    
                    class_name = CLASS_NAMES[class_id]
                    
                    # Add row to manifest
                    manifest_rows.append({
                        'slide_id': biopsia,
                        'tile_path': str(tile_path.absolute()),
                        'mask_path': str(mask_path.absolute()),
                        'instance_id': instance_id,
                        'class_name': class_name,
                        'class_id': class_id,
                        'gray_value': int(dominant),
                        'bbox_r1': int(r1),
                        'bbox_c1': int(c1),
                        'bbox_r2': int(r2),
                        'bbox_c2': int(c2),
                        'split': split_name,
                    })
    
    # Build DataFrame
    df = pd.DataFrame(manifest_rows)
    
    # Validation
    if df.empty:
        warnings.warn("No glomerulus instances found. Manifest is empty.")
        return df
    
    # Check for NaN in critical columns
    critical_cols = ['slide_id', 'tile_path', 'mask_path', 'instance_id', 
                     'class_id', 'gray_value', 'split']
    for col in critical_cols:
        if col not in df.columns:
            raise ValueError(f"Missing critical column: {col}")
        if df[col].isna().any():
            raise ValueError(f"Found NaN values in critical column: {col}")
    
    # Log statistics
    split_counts = df['split'].value_counts()
    print(f"\nGlomerulus Manifest Summary:")
    print(f"  Total instances: {len(df)}")
    print(f"  Split distribution:")
    for split in ['train', 'val', 'test']:
        count = split_counts.get(split, 0)
        pct = 100.0 * count / len(df) if len(df) > 0 else 0.0
        print(f"    {split}: {count} ({pct:.1f}%)")
    
    class_counts = df['class_id'].value_counts().sort_index()
    print(f"  Class distribution:")
    for class_id in sorted(class_counts.index):
        count = class_counts[class_id]
        pct = 100.0 * count / len(df)
        class_name = CLASS_NAMES[class_id]
        print(f"    {class_id} ({class_name}): {count} ({pct:.1f}%)")
    
    # Save to CSV if requested
    if output_csv is not None:
        output_path = Path(output_csv)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(output_path, index=False)
        print(f"\n  ✓ Manifest saved to {output_path}")
    
    return df


In [5]:
def validate_manifest(manifest_df: pd.DataFrame, output_dir: str = 'Salidas/Clasificador'):
    """
    Validate manifest: print distribution, warnings, and visual grid.
    """
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from matplotlib.gridspec import GridSpec
    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # === 1. Distribution table ===
    print("\n=== MANIFEST DISTRIBUTION ===")
    print(f"Total instances: {len(manifest_df)}")
    
    # By class and split
    print("\nBy class and split:")
    dist_table = pd.crosstab(
        manifest_df['class_name'],
        manifest_df['split'],
        margins=True
    )
    print(dist_table)
    
    # By biopsy and class
    print("\nTop 10 biopsies by instance count:")
    biopsy_counts = manifest_df['slide_id'].value_counts().head(10)
    print(biopsy_counts)
    
    # === 2. Check for class imbalance warnings ===
    for split in ['train', 'val', 'test']:
        split_df = manifest_df[manifest_df['split'] == split]
        for cls in CLASS_NAMES:
            count = len(split_df[split_df['class_name'] == cls])
            if count == 0:
                print(f"⚠️  WARNING: {cls} has 0 instances in {split} split!")
            elif count < 5:
                print(f"⚠️  WARNING: {cls} has only {count} instances in {split} split")
    
    # === 3. Visual grid: N=8 crops per class ===
    fig = plt.figure(figsize=(20, 12))
    gs = GridSpec(len(CLASS_NAMES), 8, figure=fig, hspace=0.4, wspace=0.3)
    
    for class_idx, class_name in enumerate(CLASS_NAMES):
        class_df = manifest_df[manifest_df['class_name'] == class_name].head(8)
        
        for crop_idx, (_, row) in enumerate(class_df.iterrows()):
            ax = fig.add_subplot(gs[class_idx, crop_idx])
            
            try:
                # Load RGB
                rgb = load_rgb_image(row['tile_path'])
                r1, c1, r2, c2 = int(row['bbox_r1']), int(row['bbox_c1']), int(row['bbox_r2']), int(row['bbox_c2'])
                crop_rgb = rgb[r1:r2, c1:c2]
                
                # Display
                ax.imshow(crop_rgb)
                ax.set_title(f"{class_name}\n{row['slide_id'][:15]}\narea={row.get('area_px', '?')}", fontsize=8)
                ax.axis('off')
            except Exception as e:
                ax.text(0.5, 0.5, f"Error: {str(e)[:20]}", ha='center', va='center', fontsize=8)
                ax.axis('off')
    
    # Save figure
    fig_path = output_dir / 'manifest_visual_grid.png'
    plt.savefig(fig_path, dpi=100, bbox_inches='tight')
    plt.close()
    print(f"\n✓ Saved visual grid to {fig_path}")
    
    # === 4. Compute area statistics ===
    if 'area_px' not in manifest_df.columns:
        # Compute area from bbox if not present
        manifest_df['area_px'] = (manifest_df['bbox_r2'] - manifest_df['bbox_r1']) * (manifest_df['bbox_c2'] - manifest_df['bbox_c1'])
    
    print("\nArea statistics (pixels) by class:")
    area_stats = manifest_df.groupby('class_name')['area_px'].agg(['min', 'mean', 'max', 'count'])
    print(area_stats)
    
    return manifest_df

In [ ]:
class GlomeruliClassificationDataset(Dataset):
    """Carga instancias de glomérulos desde manifest pandas.
    
    Extrae crops on-the-fly vía bbox+margin, aplica augmentations,
    retorna tensores 4-channel [RGB + máscara].
    """
    
    def __init__(
        self,
        manifest_df: pd.DataFrame,
        images_dir: str,
        masks_dir: str,
        input_size: int = 224,
        mask_size: int = 224,
        margin_ratio: float = 0.25,
        split: str = 'train',
        config: dict = None,
        channel_means: np.ndarray = None,
        channel_stds: np.ndarray = None,
        reinhard: dict = None,
    ):
        self.manifest_df = manifest_df[manifest_df['split'] == split].reset_index(drop=True)
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.input_size = input_size
        self.mask_size = mask_size
        self.margin_ratio = margin_ratio
        self.split = split
        self.config = config or {}
        self.channel_means = channel_means
        self.channel_stds = channel_stds
        self.reinhard = reinhard
        
        # Augmentations solo para train
        self.use_augment = (split == 'train')
        if self.use_augment:
            self.transform = A.Compose([
                A.ElasticTransform(p=0.10, border_mode=cv2.BORDER_REFLECT),
                A.GridDistortion(p=0.10, border_mode=cv2.BORDER_REFLECT),
                A.HEStain(p=0.25),
                A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
                A.GaussNoise(p=0.1),
                A.CoarseDropout(max_holes=1, max_height=32, max_width=32, p=0.1),
            ], additional_targets={'mask': 'mask'})
        else:
            self.transform = None
    
    def __len__(self):
        return len(self.manifest_df)
    
    def __getitem__(self, idx):
        row = self.manifest_df.iloc[idx]
        
        # Cargar imagen RGB (tile completo)
        tile_path = row['tile_path']
        rgb = load_rgb_image(tile_path)
        
        # Cargar máscara multiclase
        mask_path = row['mask_path']
        mask_mc = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
        # Reconstruir bbox desde columnas separadas (FIX: bbox_r1, bbox_c1, bbox_r2, bbox_c2)
        bbox = (int(row['bbox_r1']), int(row['bbox_c1']), int(row['bbox_r2']), int(row['bbox_c2']))
        class_id = row['class_id']
        
        # Compute margin
        h, w = bbox[2] - bbox[0], bbox[3] - bbox[1]
        margin_h = int(h * self.margin_ratio)
        margin_w = int(w * self.margin_ratio)
        
        # Crop
        r_min = max(0, bbox[0] - margin_h)
        r_max = min(rgb.shape[0], bbox[2] + margin_h)
        c_min = max(0, bbox[1] - margin_w)
        c_max = min(rgb.shape[1], bbox[3] + margin_w)
        
        rgb_crop = rgb[r_min:r_max, c_min:c_max]
        mask_crop = mask_mc[r_min:r_max, c_min:c_max]
        
        # Binarizar máscara
        binary_mask = (mask_crop > 0).astype(np.uint8)
        
        # Perturbar en train
        if self.use_augment:
            binary_mask = perturb_mask(binary_mask)
        
        # Resize
        rgb_crop = cv2.resize(rgb_crop, (self.input_size, self.input_size), interpolation=cv2.INTER_LINEAR)
        binary_mask = cv2.resize(binary_mask, (self.mask_size, self.mask_size), interpolation=cv2.INTER_NEAREST)
        
        # Preprocesar RGB
        rgb_prep = preprocess_rgb_image(
            rgb_crop,
            channel_means=self.channel_means,
            channel_stds=self.channel_stds,
            reinhard=self.reinhard,
        )
        
        # Augmentations
        if self.transform is not None:
            augmented = self.transform(image=rgb_prep, mask=binary_mask)
            rgb_prep = augmented['image']
            binary_mask = augmented['mask']
        
        # Tensores
        rgb_tensor = torch.from_numpy(np.transpose(rgb_prep, (2, 0, 1))).float()
        mask_tensor = torch.from_numpy(binary_mask[np.newaxis, :, :]).float()
        
        # Concatenar: [R, G, B, mask]
        image_4ch = torch.cat([rgb_tensor, mask_tensor], dim=0)
        
        return {
            'image': image_4ch,
            'label': torch.tensor(class_id, dtype=torch.long),
            'instance_id': row['instance_id'],
        }

    @staticmethod
    def get_sampling_weights(manifest_df: pd.DataFrame, split: str = 'train'):
        subset = manifest_df[manifest_df['split'] == split]
        class_counts = subset['class_id'].value_counts().to_dict()
        weights = np.array([1.0 / np.sqrt(class_counts.get(cid, 1)) for cid in subset['class_id']])
        return weights / weights.sum()

In [7]:
def build_classifier(backbone: str = 'efficientnet_b0', num_classes: int = 4, pretrained: bool = True):
    """Crea clasificador multiclase con timm — adapta automáticamente 4 canales en conv1."""
    model = timm.create_model(backbone, in_chans=4, num_classes=num_classes, pretrained=pretrained)
    return model

In [8]:
def create_classification_dataloaders(
    dataset_train: GlomeruliClassificationDataset,
    dataset_val: GlomeruliClassificationDataset,
    dataset_test: GlomeruliClassificationDataset,
    config: dict,
):
    """Crea dataloaders con WeightedRandomSampler para train."""
    batch_size = config.get('batch_size', 32)
    
    # Train: WeightedRandomSampler
    weights_train = GlomeruliClassificationDataset.get_sampling_weights(
        dataset_train.manifest_df, split='train'
    )
    sampler_train = WeightedRandomSampler(weights_train, len(weights_train), replacement=True)
    loader_train = DataLoader(dataset_train, batch_size=batch_size, sampler=sampler_train, num_workers=2, pin_memory=True)
    
    # Val / Test
    loader_val = DataLoader(dataset_val, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    loader_test = DataLoader(dataset_test, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    
    return loader_train, loader_val, loader_test

In [9]:
def build_criterion(manifest_df: pd.DataFrame, sampling_strategy: str = 'sampler_balanced_ce_unweighted'):
    """Crea loss criterion según estrategia."""
    
    if 'focal' in sampling_strategy.lower():
        from torchvision.ops.focal_loss import sigmoid_focal_loss
        return sigmoid_focal_loss
    elif 'weighted' in sampling_strategy.lower():
        counts = manifest_df[manifest_df['split'] == 'train']['class_id'].value_counts().sort_index()
        weights = 1.0 / np.sqrt(counts.values)
        weights = weights / weights.sum() * len(counts)
        return nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=device))
    else:
        return nn.CrossEntropyLoss()

In [ ]:
def _train_single_fold(
    fold_name: str,
    fold_biopsias: dict,
    manifest_full: pd.DataFrame,
    config: dict,
    preprocessing_params: dict,
) -> dict:
    """
    Train classifier on a single fold.
    Returns dict with metrics.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    output_dir = Path(config['output_dir'])
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Filter manifest for this fold
    fold_mask = manifest_full['slide_id'].isin(fold_biopsias['train'] + fold_biopsias['val'])
    train_val_manifest = manifest_full[fold_mask].copy()
    
    # Further split into train/val within the fold
    train_mask = train_val_manifest['slide_id'].isin(fold_biopsias['train'])
    dataset_train = GlomeruliClassificationDataset(
        train_val_manifest[train_mask],
        config=config,
        split='train',
        **preprocessing_params,
    )
    dataset_val = GlomeruliClassificationDataset(
        train_val_manifest[~train_mask],
        config=config,
        split='val',
        **preprocessing_params,
    )
    
    # Test set (from fold_biopsias['test'])
    test_mask = manifest_full['slide_id'].isin(fold_biopsias['test'])
    dataset_test = GlomeruliClassificationDataset(
        manifest_full[test_mask],
        config=config,
        split='test',
        **preprocessing_params,
    )
    
    # Create dataloaders
    train_loader, val_loader, test_loader = create_classification_dataloaders(
        dataset_train, dataset_val, dataset_test, config
    )
    
    # Build model, optimizer, scheduler
    model = build_classifier(config).to(device)
    criterion = build_criterion(config, dataset_train)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.get('lr', 1e-3),
        weight_decay=config.get('weight_decay', 1e-4),
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0=5,
        T_mult=2,
        eta_min=1e-6,
    )
    
    # Training loop
    max_epochs = config.get('max_epochs', 60)
    patience = config.get('early_stopping_patience', 10)
    best_balanced_acc = 0
    patience_counter = 0
    
    print(f"\n{'='*80}")
    print(f"Training {fold_name} | Epochs: {max_epochs}, Patience: {patience}, LR: {config.get('lr', 1e-3)}")
    print(f"  Train: {len(dataset_train)} | Val: {len(dataset_val)} | Test: {len(dataset_test)}")
    print(f"{'='*80}")
    
    for epoch in range(max_epochs):
        train_loss, train_metrics = train_epoch_clf(model, train_loader, criterion, optimizer, device, config)
        val_loss, val_metrics = eval_epoch_clf(model, val_loader, criterion, device)
        val_balanced_acc = val_metrics['balanced_accuracy']
        
        new_best = False
        if val_balanced_acc > best_balanced_acc:
            best_balanced_acc = val_balanced_acc
            patience_counter = 0
            # Save checkpoint
            ckpt_path = output_dir / f'best_model_{fold_name}.pth'
            torch.save(model.state_dict(), ckpt_path)
            new_best = True
        else:
            patience_counter += 1
        
        # Print epoch summary
        status = "★ best" if new_best else f"wait {patience_counter}/{patience}"
        print(
            f"Epoch {epoch+1:3d}/{max_epochs} | "
            f"Loss: {train_loss:.4f}/{val_loss:.4f} | "
            f"Acc: {train_metrics['balanced_accuracy']:.3f}/{val_balanced_acc:.3f} | "
            f"{status}"
        )
        
        if patience_counter >= patience:
            print(f"→ Early stopping at epoch {epoch+1}")
            break
        
        scheduler.step()
    
    # Final evaluation on test set
    model.load_state_dict(torch.load(output_dir / f'best_model_{fold_name}.pth'))
    test_loss, test_metrics = eval_epoch_clf(model, test_loader, criterion, device)
    
    print(f"✓ {fold_name} complete | Test Acc: {test_metrics['balanced_accuracy']:.4f}")
    
    return test_metrics

In [ ]:
def train_classifier(config: dict):
    """
    Entrena el clasificador multiclase de glomérulos.
    Retorna (best_model_weights, history_dict, manifest_df).
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    output_dir = Path(config['output_dir'])
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # 1. Load U-Net checkpoint para stats
    checkpoints_dir = Path('checkpoints')
    unet_checkpoints = sorted(checkpoints_dir.glob('*_best.pth'), key=lambda p: p.stat().st_mtime, reverse=True)
    if not unet_checkpoints:
        raise FileNotFoundError(f"No U-Net checkpoint found in {checkpoints_dir}")
    
    unet_ckpt = torch.load(unet_checkpoints[0], map_location=device)
    channel_means = unet_ckpt['channel_means']
    channel_stds = unet_ckpt['channel_stds']
    reinhard_stats = unet_ckpt.get('reinhard_stats', None)
    
    print(f"\nLoading U-Net checkpoint: {unet_checkpoints[0].name}")
    print(f"  ✓ Channel means: {channel_means}")
    print(f"  ✓ Channel stds: {channel_stds}")
    
    # 2. Split biopsias
    train_biopsias, val_biopsias, test_biopsias, biopsias_dict = split_biopsias(
        config['images_dir'],
        train_size=config['train_size'],
        val_size=config['val_size'],
        seed=config['seed'],
    )
    split_result = {'train': train_biopsias, 'val': val_biopsias, 'test': test_biopsias}
    print(f"✓ Split biopsias: train={len(split_result['train'])}, val={len(split_result['val'])}, test={len(split_result['test'])}")
    
    # 3. Build manifest
    manifest_df = build_glomerulus_manifest(
        images_dir=config['images_dir'],
        masks_dir=config['masks_dir'],
        split_biopsias_result=split_result,
        min_area_px=config['min_area_px'],
        min_distance=config['min_distance'],
        output_csv=os.path.join(config['output_dir'], 'manifest.csv'),
    )
    
    # 4. Validate manifest
    manifest_df = validate_manifest(manifest_df, config['output_dir'])
    
    # 5. Setup preprocessing params
    preprocessing_params = {
        'images_dir': config['images_dir'],
        'masks_dir': config['masks_dir'],
        'input_size': config['input_size'],
        'mask_size': config['mask_size'],
        'margin_ratio': config['margin_ratio'],
        'channel_means': channel_means,
        'channel_stds': channel_stds,
        'reinhard': reinhard_stats,
    }
    
    # 6. Train (single fold for now)
    fold_biopsias = {
        'train': train_biopsias,
        'val': val_biopsias,
        'test': test_biopsias,
    }
    
    test_metrics = _train_single_fold(
        fold_name='default',
        fold_biopsias=fold_biopsias,
        manifest_full=manifest_df,
        config=config,
        preprocessing_params=preprocessing_params,
    )
    
    # Load best model
    best_model_path = output_dir / 'best_model_default.pth'
    model = build_classifier(config).to(device)
    model.load_state_dict(torch.load(best_model_path))
    
    history = {'best_balanced_acc': test_metrics['balanced_accuracy']}
    
    print(f"\n{'='*80}")
    print(f"TRAINING COMPLETE")
    print(f"  Best test balanced_accuracy: {test_metrics['balanced_accuracy']:.4f}")
    print(f"{'='*80}\n")
    
    return model, history, manifest_df

In [ ]:
# === VIZ-B: Augmentations showcase - 1 glomerulo + 8 variantes ===
print("VIZ-B: Augmentation variants of a single glomerulus")

# Pick one random glomerulus from train split
train_manifest = manifest_df[manifest_df['split'] == 'train']
if len(train_manifest) > 0:
    sample_idx = random.randint(0, len(train_manifest) - 1)
    row = train_manifest.iloc[sample_idx]
    
    # Load and prep
    rgb = load_rgb_image(row['tile_path'])
    mask_mc = cv2.imread(row['mask_path'], cv2.IMREAD_GRAYSCALE)
    bbox = (int(row['bbox_r1']), int(row['bbox_c1']), int(row['bbox_r2']), int(row['bbox_c2']))
    
    h, w = bbox[2] - bbox[0], bbox[3] - bbox[1]
    margin_h = int(h * 0.25)
    margin_w = int(w * 0.25)
    r_min = max(0, bbox[0] - margin_h)
    r_max = min(rgb.shape[0], bbox[2] + margin_h)
    c_min = max(0, bbox[1] - margin_w)
    c_max = min(rgb.shape[1], bbox[3] + margin_w)
    
    crop_rgb = rgb[r_min:r_max, c_min:c_max].astype(np.uint8)
    crop_mask = mask_mc[r_min:r_max, c_min:c_max]
    crop_mask = (crop_mask > 0).astype(np.uint8)
    
    # Resize
    crop_rgb = cv2.resize(crop_rgb, (224, 224))
    crop_mask = cv2.resize(crop_mask, (224, 224), interpolation=cv2.INTER_NEAREST)
    
    # Build augmentation transform
    aug_transform = A.Compose([
        A.ElasticTransform(p=1.0, border_mode=cv2.BORDER_REFLECT),
        A.GridDistortion(p=1.0, border_mode=cv2.BORDER_REFLECT),
        A.HEStain(p=1.0),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=1.0),
        A.GaussNoise(p=1.0),
        A.CoarseDropout(max_holes=1, max_height=32, max_width=32, p=1.0),
    ], additional_targets={'mask': 'mask'})
    
    # Generate 8 augmented variants
    fig, axes = plt.subplots(1, 9, figsize=(20, 2.5))
    
    # Original
    img_orig = make_mask_overlay(crop_rgb, crop_mask, color=(0, 255, 0), alpha=0.3)
    axes[0].imshow(img_orig)
    axes[0].set_title("Original", fontsize=9, fontweight='bold')
    axes[0].axis('off')
    
    # 8 augmented variants
    for aug_idx in range(8):
        augmented = aug_transform(image=crop_rgb, mask=crop_mask)
        aug_rgb = augmented['image']
        aug_mask = augmented['mask']
        
        img_aug = make_mask_overlay(aug_rgb, aug_mask, color=(0, 255, 0), alpha=0.3)
        axes[aug_idx + 1].imshow(img_aug)
        axes[aug_idx + 1].set_title(f"Aug {aug_idx+1}", fontsize=9)
        axes[aug_idx + 1].axis('off')
    
    plt.suptitle(f"{row['class_name']} | {row['slide_id']}", fontsize=11, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('Salidas/Clasificador/viz_augmentations.png', dpi=100, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved to Salidas/Clasificador/viz_augmentations.png")
else:
    print("⚠️  No train samples available")

In [ ]:
# === VIZ-A: 10 glomerulos del dataset con sus máscaras ===
print("VIZ-A: 10 reconstructed glomeruli with masks")

# Asume que manifest_df existe (creado durante training)
# Toma 10 random glomerulos de todo el dataset
random.seed(42)
sample_indices = random.sample(range(len(manifest_df)), min(10, len(manifest_df)))

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes = axes.flatten()

for plot_idx, df_idx in enumerate(sample_indices):
    row = manifest_df.iloc[df_idx]
    
    try:
        # Load RGB
        rgb = load_rgb_image(row['tile_path'])
        bbox = (int(row['bbox_r1']), int(row['bbox_c1']), int(row['bbox_r2']), int(row['bbox_c2']))
        
        # Extract crop sin margen (para mostrar el glomerulo)
        h, w = bbox[2] - bbox[0], bbox[3] - bbox[1]
        margin_h = int(h * 0.25)
        margin_w = int(w * 0.25)
        r_min = max(0, bbox[0] - margin_h)
        r_max = min(rgb.shape[0], bbox[2] + margin_h)
        c_min = max(0, bbox[1] - margin_w)
        c_max = min(rgb.shape[1], bbox[3] + margin_w)
        
        crop_rgb = rgb[r_min:r_max, c_min:c_max]
        crop_rgb = cv2.resize(crop_rgb, (224, 224))
        
        # Load mask
        mask_mc = cv2.imread(row['mask_path'], cv2.IMREAD_GRAYSCALE)
        mask_crop = mask_mc[r_min:r_max, c_min:c_max]
        mask_crop = cv2.resize(mask_crop, (224, 224), interpolation=cv2.INTER_NEAREST)
        binary_mask = (mask_crop > 0).astype(np.uint8)
        
        # Overlay mask
        img_with_mask = make_mask_overlay(crop_rgb, binary_mask, color=(0, 255, 0), alpha=0.3)
        
        axes[plot_idx].imshow(img_with_mask)
        axes[plot_idx].set_title(
            f"{row['class_name']}\n{row['slide_id'][:12]} | area={int((bbox[2]-bbox[0])*(bbox[3]-bbox[1]))}px",
            fontsize=9
        )
        axes[plot_idx].axis('off')
    except Exception as e:
        axes[plot_idx].text(0.5, 0.5, f"Error:\n{str(e)[:30]}", ha='center', va='center', fontsize=8)
        axes[plot_idx].axis('off')

plt.tight_layout()
plt.savefig('Salidas/Clasificador/viz_10_glomeruli.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved to Salidas/Clasificador/viz_10_glomeruli.png")

In [ ]:
# 5. Train classifier
print("\n" + "="*80)
print("TRAINING CLASSIFIER")
print("="*80)

model, train_history, manifest_df = train_classifier(CONFIG)

print(f"\n✓ Training complete!")
print(f"  Best balanced_acc: {train_history['best_balanced_acc']:.4f}")
print(f"  Manifest instances: {len(manifest_df)}")
print(f"    Train: {len(manifest_df[manifest_df['split']=='train'])}")
print(f"    Val:   {len(manifest_df[manifest_df['split']=='val'])}")
print(f"    Test:  {len(manifest_df[manifest_df['split']=='test'])}")

In [ ]:
# === VIZ-C: Predicciones vs Ground Truth (20 crops) ===
print("VIZ-C: Predictions vs Ground Truth")

# Asume que model está disponible del training anterior
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Create test dataset y loader
preprocessing_params = {
    'images_dir': CONFIG['images_dir'],
    'masks_dir': CONFIG['masks_dir'],
    'input_size': CONFIG['input_size'],
    'mask_size': CONFIG['mask_size'],
    'margin_ratio': CONFIG['margin_ratio'],
    'channel_means': channel_means,
    'channel_stds': channel_stds,
    'reinhard': reinhard_stats,
}

test_manifest = manifest_df[manifest_df['split'] == 'test']
dataset_test_viz = GlomeruliClassificationDataset(
    test_manifest,
    config=CONFIG,
    split='test',
    **preprocessing_params,
)
loader_test_viz = DataLoader(dataset_test_viz, batch_size=1, shuffle=False, num_workers=0)

# Run inference on test set
model.eval()
all_predictions = []
all_labels = []
all_probs = []

with torch.no_grad():
    for batch in tqdm(loader_test_viz, desc="Inferencing", leave=False):
        x, y = batch['image'].to(device), batch['label'].to(device)
        logits = model(x)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(logits, dim=1)
        
        all_predictions.append(preds.cpu().numpy()[0])
        all_labels.append(y.cpu().numpy()[0])
        all_probs.append(probs.cpu().numpy()[0])

all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

# Get confidences
confidences = np.max(all_probs, axis=1)

# Sort by confidence (descending) to show most confident first
sorted_idx = np.argsort(-confidences)

# Pick top 20 for visualization
n_show = min(20, len(sorted_idx))
sorted_idx = sorted_idx[:n_show]

# Create grid 4x5
fig, axes = plt.subplots(4, 5, figsize=(18, 14))
axes = axes.flatten()

for plot_idx, manifest_idx in enumerate(sorted_idx):
    row = test_manifest.iloc[manifest_idx]
    pred_class_id = int(all_predictions[manifest_idx])
    label_class_id = int(all_labels[manifest_idx])
    confidence = float(confidences[manifest_idx])
    
    is_correct = (pred_class_id == label_class_id)
    
    try:
        # Load crop
        rgb = load_rgb_image(row['tile_path'])
        mask_mc = cv2.imread(row['mask_path'], cv2.IMREAD_GRAYSCALE)
        bbox = (int(row['bbox_r1']), int(row['bbox_c1']), int(row['bbox_r2']), int(row['bbox_c2']))
        
        h, w = bbox[2] - bbox[0], bbox[3] - bbox[1]
        margin_h = int(h * 0.25)
        margin_w = int(w * 0.25)
        r_min = max(0, bbox[0] - margin_h)
        r_max = min(rgb.shape[0], bbox[2] + margin_h)
        c_min = max(0, bbox[1] - margin_w)
        c_max = min(rgb.shape[1], bbox[3] + margin_w)
        
        crop_rgb = rgb[r_min:r_max, c_min:c_max]
        crop_rgb = cv2.resize(crop_rgb, (224, 224))
        
        # Add border
        border_color = (0, 255, 0) if is_correct else (255, 0, 0)
        crop_with_border = cv2.copyMakeBorder(crop_rgb, 3, 3, 3, 3, cv2.BORDER_CONSTANT, value=border_color)
        
        axes[plot_idx].imshow(crop_with_border)
        
        pred_name = CLASS_NAMES[pred_class_id]
        label_name = CLASS_NAMES[label_class_id]
        
        axes[plot_idx].set_title(
            f"Pred: {pred_name}\nGT: {label_name}\n{confidence:.0%}",
            fontsize=9,
            color='green' if is_correct else 'red',
            fontweight='bold' if is_correct else 'normal'
        )
        axes[plot_idx].axis('off')
    except Exception as e:
        axes[plot_idx].text(0.5, 0.5, f"Error:\n{str(e)[:25]}", ha='center', va='center', fontsize=8)
        axes[plot_idx].axis('off')

plt.suptitle(f"Predictions vs Ground Truth (sorted by confidence)\nGreen border=Correct, Red border=Error", 
             fontsize=12, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('Salidas/Clasificador/viz_predictions_vs_gt.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved to Salidas/Clasificador/viz_predictions_vs_gt.png")

In [12]:
# ============================================================================
# MAIN EXECUTION: Load checkpoint & Build Manifest
# ============================================================================

from pathlib import Path

# 1. Find and load U-Net checkpoint (most recent *_best.pth)
checkpoints_dir = Path('checkpoints')
unet_checkpoints = sorted(checkpoints_dir.glob('*_best.pth'), key=lambda p: p.stat().st_mtime, reverse=True)

if not unet_checkpoints:
    raise FileNotFoundError(f"No U-Net checkpoint (*_best.pth) found in {checkpoints_dir}")

unet_ckpt_path = unet_checkpoints[0]
print(f"Loading U-Net checkpoint: {unet_ckpt_path.name}")

unet_ckpt = torch.load(unet_ckpt_path, map_location=device)
channel_means = unet_ckpt['channel_means']
channel_stds = unet_ckpt['channel_stds']
reinhard_stats = unet_ckpt.get('reinhard_stats', None)
print(f"  ✓ Channel means: {channel_means}")
print(f"  ✓ Channel stds: {channel_stds}")

# 2. Split biopsias — retorna tupla (train_list, val_list, test_list, biopsias_dict)
train_biopsias, val_biopsias, test_biopsias, biopsias_dict = split_biopsias(
    CONFIG['images_dir'],
    train_size=CONFIG['train_size'],
    val_size=CONFIG['val_size'],
    seed=CONFIG['seed'],
)
split_result = {
    'train': train_biopsias,
    'val': val_biopsias,
    'test': test_biopsias,
}
print(f"\n✓ Split biopsias: train={len(split_result['train'])}, val={len(split_result['val'])}, test={len(split_result['test'])}")

# 3. Build manifest
manifest_df = build_glomerulus_manifest(
    images_dir=CONFIG['images_dir'],
    masks_dir=CONFIG['masks_dir'],
    split_biopsias_result=split_result,
    min_area_px=CONFIG['min_area_px'],
    min_distance=CONFIG['min_distance'],
    output_csv=os.path.join(CONFIG['output_dir'], 'manifest.csv'),
)
print(f"✓ Manifest: {len(manifest_df)} instances (train={len(manifest_df[manifest_df['split']=='train'])}, val={len(manifest_df[manifest_df['split']=='val'])}, test={len(manifest_df[manifest_df['split']=='test'])})")

Loading U-Net checkpoint: unet_binary_20260513_072214_best.pth
  ✓ Channel means: [0.5249426116573582, 0.4619087558025785, 0.4945691148938925]
  ✓ Channel stds: [0.12682547154319532, 0.15825028652078063, 0.06188548903405716]

✓ Split biopsias: train=49, val=11, test=11


KeyboardInterrupt: 

In [ ]:
# 4. Create datasets
dataset_train = GlomeruliClassificationDataset(
    manifest_df, CONFIG['images_dir'], CONFIG['masks_dir'], split='train',
    channel_means=channel_means, channel_stds=channel_stds, reinhard=reinhard_stats, config=CONFIG
)
dataset_val = GlomeruliClassificationDataset(
    manifest_df, CONFIG['images_dir'], CONFIG['masks_dir'], split='val',
    channel_means=channel_means, channel_stds=channel_stds, reinhard=reinhard_stats, config=CONFIG
)
dataset_test = GlomeruliClassificationDataset(
    manifest_df, CONFIG['images_dir'], CONFIG['masks_dir'], split='test',
    channel_means=channel_means, channel_stds=channel_stds, reinhard=reinhard_stats, config=CONFIG
)

loader_train, loader_val, loader_test = create_classification_dataloaders(
    dataset_train, dataset_val, dataset_test, CONFIG
)
print(f"✓ Dataloaders created: train={len(loader_train)}, val={len(loader_val)}, test={len(loader_test)}")

In [ ]:
# 5. Train classifier
print("\n" + "="*80)
print("TRAINING CLASSIFIER")
print("="*80)

model, train_history = train_classifier(CONFIG)

print(f"\n✓ Training complete!")
print(f"  Best balanced_acc: {train_history['best_balanced_acc']:.4f}")